# LSTM baseline on the NTM copy task

First: **Runtime → Change runtime type → T4 GPU**, then run the cells in order.

In [ ]:
# Safe to re-run: clones the repo once, then just pulls updates
%cd /content
!nvidia-smi --query-gpu=name --format=csv
!git clone https://github.com/mgupta8143/neural-turing-machines.git 2>/dev/null || git -C neural-turing-machines pull
%cd /content/neural-turing-machines

Stop any training run that's already going, so two runs don't write to the same files.

In [ ]:
!pkill -f "main.py train" || true

Start training in the background: batch size 4, learning rate 1e-4 (about 4× the paper's 3e-5, scaled up for the bigger batch). This takes roughly 1.5–2 hours for 1M sequences and logs every ~1,000 sequences.

Other options for the first line:
- Quick run, about 30 min: `!nohup python -u main.py train --batch-size 16 --learning-rate 1e-4 > train.log 2>&1 &`
- Paper's exact settings, about 6.5 hours: `!nohup python -u main.py train > train.log 2>&1 &`

In [ ]:
!nohup python -u main.py train --batch-size 4 --learning-rate 1e-4 > train.log 2>&1 &
!sleep 30 && tail -n 3 train.log

Check progress. Re-run whenever you like: it shows the latest cost and elapsed minutes.

In [ ]:
!tail -n 5 train.log

Draw Figures 3 and 5 from the latest saved model. Works any time after the first 1,000 sequences (the first log line).

In [ ]:
!python main.py plot
import os
from IPython.display import Image, display
for path in ['figures/copy_learning_curve.png', 'figures/copy_generalisation.png']:
    if os.path.exists(path):
        display(Image(path))

Try your own sequence. Edit `vectors` below: each item is 8 bits of 0/1. Or set `random_length` to a number to use a random sequence of that length (try something longer than 20 to see the LSTM fail). Works during training too, with the latest saved model.

In [ ]:
vectors = "10110010 01100101 11110000 00001111"
random_length = None  # e.g. 30

if random_length:
    !python main.py try --random {random_length}
else:
    !python main.py try {vectors}
from IPython.display import Image, display
display(Image('figures/copy_try.png'))

Optional: copy the results and figures to Google Drive, so they survive when the Colab session ends.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/ntm-results
!cp -r results figures /content/drive/MyDrive/ntm-results/